# POC 3 project (Alexnet) (COLAB)

- Creation : *24/05/2025*

**Utilisation de ConvnetWrapperForDeconvolution + DéconNet + déconvolution sur GPU**

Réalisation d'un tentative de reproduction de la Figure 2 de l'article :
1. [X] Chargement d'une version de ConvNet (AlexNet ou VGG*) adaptée pour la déconvolution
1. [X] Pour chaque couche cachée choisie dans la Figure 3 de l'article, sur (une sous-partie du)/le jeu de données
   - récupérer le top K des neurones choisis au hasard
   - récupérer le top K des activations maximums par couche, toutes cartes confondues
1. [X] Affichage sous forme de grille des déconvolutions et des champs réceptifs. Comparaison avec les images fournies en entrée.
1. [X] Calcul des déconvolutions des tops K, les champs réceptifs équivalents et les extraits des images en entrée correspondants

In [ ]:
import os
if os.path.exists("./utils"):
  !rm -rf ./utils/
!rm -f ./datasets.py
!rm -f ./imagenet_labels.py

remove_images = False
if remove_images:
  #if os.path.exists("./imagenet-sample-images-master"):
  !rm -rf ./imagenet-sample-images-master/
  !rm -rf ./imagenet_val_images/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SHARED_PROJECT_PATH = "/content/drive/MyDrive/Colab Notebooks/CNAM/RCP209/Projet/"
if not os.path.exists(SHARED_PROJECT_PATH):
  SHARED_PROJECT_PATH = "/content/drive/MyDrive/Colab Notebooks/NG4Dev Notebooks/CNAM/RCP209/Projet/"  

In [ ]:
!cp -rv "{SHARED_PROJECT_PATH}"* .
!rm -f *.ipynb
#!rm -f ./imagenet-sample-images-master.zip
#!unzip -q imagenet-sample-images-master.zip
if not os.path.exists("./imagenet_val_images"):
  !gunzip ./imagenet_val_images.tar.gz
  !mkdir ./imagenet_val_images
  !tar -xf imagenet_val_images.tar -C ./imagenet_val_images
  !rm -f ./imagenet_val_images.tar

In [ ]:
!ls ./imagenet_val_images/|wc -l

In [ ]:
import sys
sys.path.append("content/" + "utils")

print("\n".join(sys.path))

In [ ]:
#import os
#os.kill(os.getpid(), 9)

In [ ]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

## Variables de contrôle

In [ ]:
# Variables de contrôle d'éxécution du script
test_dataloader = True
test_image_one_by_one = False

# Indexes of layer selected in the paper
# 0 : sortie de la première couche opératoire du modèle
#paper_layer_idx = [2, 5, 7, 9, 12]
probed_layer_idx = [7] # couche cachées à surveiller
#probed_neuron_by_layer = [9, 16, 12, 10, 10] # neurones à surveiller dans chaque couche cachée
probed_neuron_by_layer = [12] # neurones à surveiller dans chaque couche cachée
K = 9 # Top K

batch_size = 32
SEED = 42 # Random seed for reproducibility

N_images = 2 # Number of images for image by image test
i_stop = 0 # Stop if i_stop > 0

## Modules

In [ ]:
import os
import random
from typing import Tuple
from datetime import datetime

import numpy as np
import torch
import torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm, trange
from torchinfo import summary
from PIL import Image, ImageDraw

from datasets import imagenet_mean, imagenet_std, DATASET_2, DATASET_1, CustomImageDataset, get_label_data_from_filename
from utils.convnet_wrapper_for_deconvolution import ConvnetWrapperForDeconvolution
from utils.utils_cnn import get_output_sizes, get_receptive_field_in_pixel_space
from utils.utils_images import display_image_tensor as display_image_tensor_, to_0_255, unnormalize, display_images_list_grid, to_0_1, show_image_tensor, to_0_Vmax
from utils.topk import TopK
from utils.deconvnet import Deconvnet
from utils.utils_rgb import display_rgb_distributions


In [ ]:
model_name = "alexnet"
TORCHVISION_MODELS_WEIGHTS = torchvision.models.AlexNet_Weights

In [ ]:
random.seed(SEED)
#rng = np.random.default_rng()

#random_layer_count = rng.integers(len(model_deconv.convnet_features)+1)
#random_layer_idx = rng.choice(len(model_deconv.convnet_features), random_layer_count, replace=False)

In [ ]:
# Spécifiquement pour un carnet de type Jupyter
def display_image_tensor(
        img_tensor, resize: Tuple[int, int] = None, resample: int = Image.Resampling.NEAREST, verbose=True
        ):
    if display:
        display_image_tensor_(img_tensor, resize=resize, resample=resample, verbose=verbose, fn_display=display)

## Devices

Détection du GPU disponible.

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"{device} est disponible")

## Les données

Création du `dataset`et du `dataloader`.

In [ ]:
#DATASET = DATASET_2 # le 1K images pour l'instant
DATASET = DATASET_1 # le 50K images

imagenet_mean = DATASET["means"]
imagenet_std = DATASET["stds"]

geo_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
])
"""
transforms = T.Compose([
    geo_transforms,
    T.Lambda(lambda t: t/255.), # because read_image -> [0..255]
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
])
"""

transforms = TORCHVISION_MODELS_WEIGHTS.IMAGENET1K_V1.transforms()

get_label_data = lambda f: get_label_data_from_filename(f, DATASET["path"])
dataset_path = DATASET["mounted_path"] if os.path.exists(DATASET["mounted_path"]) else DATASET["path"]
print(f"Utilisation du dataset {DATASET['name']} situé dans {dataset_path}")

dataset = CustomImageDataset(
    dataset_path,
    transform=transforms,
    extension="JPEG",
    dataset_mode=True,
    only_label_idx=False, # On a besoin de l'index pour le nom du fichier
    get_label_data=get_label_data,
    )

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

En mode test : vérification de la sortie du dataset et du dataloader

In [ ]:
if test_dataloader:
    data = dataset[1]
    print(f"data: {data}")

In [ ]:
if test_dataloader:
    # Dans la configuration (dataset_mode=True, only_label_idx=False) => 
    #   [[image_trfm], [label_idx], [label_code], [idx]]
    i_element = 1
    batch = next(iter(dataloader))

    print("Type batch :", type(batch))
    print("Dimension batch:", len(batch), end="\n\n")
    
    print("Première partie du Batch :", batch[0].size())
    print("Deuxième partie du Batch :", batch[1].size())
    print("Troisième partie du Batch :", len(batch[2]))
    print("Quatrième partie du Batch :", batch[3].size(), end="\n\n")
    
    print("Image (taille du tenseur):", batch[0][i_element].size())
    print("Idx etiquette :", batch[1][i_element].item())
    print("Code etiquette :", batch[2][i_element])
    print("Id file :", batch[3][i_element].item())

## Le modèle

In [ ]:
model_convnet = torch.hub.load('pytorch/vision', model_name, weights=TORCHVISION_MODELS_WEIGHTS.IMAGENET1K_V1)
model_convnet.eval()
model_deconv = ConvnetWrapperForDeconvolution(model_convnet, model_convnet.features)

Résumé de la structure du modèle de type ConvNet

In [ ]:
#print("> Résumé de la structure du modèle de type ConvNet")
print(model_convnet)

Tableau des dimensionnalités des couches cachées du modèle

In [ ]:
# print("> Tableau des dimensionnalités des couches cachées du modèle")
batch_input = dataset[0][0].unsqueeze(dim=0)
print(batch_input.size())
model_deconv.set_return_switch_indices(False)
summary(model_convnet, input_size=batch_input.size(), mode="eval")

## Récupération des activations des neurones surveillés

Création des structures topK et choix aléatoire des coordonnées de $N$ neurones surveillés par couche cachée indiquée.

In [ ]:
# Récupération des tailles des sorties afin de pouvoir choisir des coordonnées aléatoires
# dans les sorties des couches
input_size=batch_input.size()
model_deconv.set_return_switch_indices(False)
output_sizes = get_output_sizes(model_deconv.convnet_features, input_size=input_size, last_2d=False)
print("Dimensions des différentes couches cachées")
for layer_idx, output_size in enumerate(output_sizes):
    print(f"\tcouche {layer_idx:2d} : {output_size}")

topk_activations_by_neuron = {}
coord_activations = {}
for layer_idx, n_neurons in zip(probed_layer_idx, probed_neuron_by_layer):
    coord_activations[layer_idx] = []
    i = 0
    # Randomly select N coordinates in the output of the layer : chanel, row, col
    while i < n_neurons:
        # Choisir une coordonnée aléatoire dans la sortie de la couche
        #chn = rng.integers(output_sizes[layer_idx][0])
        #row = rng.integers(output_sizes[layer_idx][1])
        #col = rng.integers(output_sizes[layer_idx][2])
        chn = random.randint(0, output_sizes[layer_idx][0]-1)
        row = random.randint(0, output_sizes[layer_idx][1]-1)
        col = random.randint(0, output_sizes[layer_idx][2]-1)
        # Vérifier que la coordonnée n'est pas déjà choisie (on ne veut pas de doublons)
        if (chn, row, col) not in coord_activations[layer_idx]:
            coord_activations[layer_idx].append((chn, row, col))
            i += 1
    topk_activations_by_neuron[layer_idx] = {coord: TopK(K) for coord in coord_activations[layer_idx]}
print(f"Coordonnées choisies : {coord_activations}")

Générations des activations

In [ ]:
### TEST "image par image###
if test_image_one_by_one:
    dataset_test = CustomImageDataset(
        dataset_path,
        transform=transforms,
        extension="JPEG",
        dataset_mode=False,
        only_label_idx=False, # On a besoin de l'index pour le nom du fichier
        get_label_data=get_label_data,
        )

    model_deconv.to(device)
    all_activations = []
    for i in trange(0, N_images):
        image, image_trfm, label_idx, label_code, input_file_idx = dataset_test[i]
        batch_input = image_trfm.unsqueeze(dim=0).to(device)
        activations = model_deconv.get_activations(batch_input, coord_activations, verbose=True)
        all_activations.append((activations, torch.tensor([input_file_idx])))

    print(f"Toutes les activations : {all_activations}")

In [ ]:
if not test_image_one_by_one:
    BATCH_IMAGES_TRFM = 0
    #BATCH_LABEL_INDICES = 1
    BATCH_FILE_INDICES = 3

    if i_stop < 0:
        i_stop = min(len(dataloader) + i_stop, 1)
    total = min(i_stop, len(dataloader)) if i_stop else len(dataloader)

    model_deconv.to(device)
    all_activations = []
    i_batch = 0
    for batch in tqdm(dataloader, total=total, desc="Propagation", unit="batch"):
        if i_stop and i_batch >= i_stop:
            break
        batch_input = batch[BATCH_IMAGES_TRFM].to(device)
        #batch_label_idx = batch[BATCH_LABEL_INDICES]
        batch_input_file_idx = batch[BATCH_FILE_INDICES]

        # On fait passer le batch dans le modèle
        # et on récupère les activations des couches sélectionnées
        #with torch.no_grad():
        #    model_deconv.convnet_features_to_device(device)

        activations = model_deconv.get_activations(batch_input, coord_activations, verbose=False)
        all_activations.append((activations, batch_input_file_idx))

        i_batch += 1

Injection dans les topK

In [ ]:
for activations, batch_input_file_idx in all_activations:
    for idx_layer, activations_layer in activations.items():
            # On fait le tri pour chaque coordonnée
            for coord, batch_activations_coord in zip(coord_activations[idx_layer], activations_layer):
                topk_activations_by_neuron[idx_layer][coord].append(batch_activations_coord.tolist(), batch_input_file_idx.tolist())

In [ ]:
all_max_topk = {}
for idx_layer, coords in topk_activations_by_neuron.items():
    max_topk = 0
    max_topk_file_idx = None
    max_coord = None
    for coord, topk in coords.items():
        print(f"> Layer {idx_layer} neuron {coord} :", topk)
        if topk[0][0] > max_topk:
            max_topk = topk[0][0]
            max_topk_file_idx = topk[0][1]
            max_coord = coord
    all_max_topk[idx_layer] = (max_topk, max_topk_file_idx, max_coord)

In [ ]:
for idx_layer, max_topk in all_max_topk.items():
    max_topk, max_topk_file_idx, max_coord = max_topk
    print(f"Layer {idx_layer} neuron {max_coord} : {max_topk} File index : {max_topk_file_idx}")

## Déconvolution

In [ ]:
deconvnet = Deconvnet(model_deconv, flip_kernels=False, use_bias=False)
print(deconvnet)

### Utils

In [ ]:
pil_to_tensor = T.PILToTensor()

In [ ]:
def merge_input_and_deconv(background_image, output_deconv, receptive_field, alpha=0.75):
    assert alpha >= 0 and alpha <= 1, "Alpha must be between 0 and 1"

    ((top, left), (bottom, right)) = receptive_field
    background_image = T.functional.to_pil_image(background_image)
    output_deconv = T.functional.to_pil_image(output_deconv).crop((left, top, right, bottom))

    # Ajouter couche alpha pour superposer l'image de sortie sur l'image de fond
    background_image = background_image.convert("RGBA")
    output_resized = output_deconv.convert("RGBA")
    
    # Créer un masque pour la transparence (optionnel)
    mask = Image.new("L", output_resized.size, int(255 * alpha))

    # Coller l'image de sortie sur l'image de fond
    background_image.paste(output_resized, (left, top), mask)

    # Afficher l'image résultante
    #display(background_image)
    return background_image

In [ ]:
inv_normalize = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.255],
    std=[1/0.229, 1/0.224, 1/0.255]
)

### Pour une image

Image originale avant redimensionnement

In [ ]:
idx_layer, coords = list(topk_activations_by_neuron.items())[0]
coord, topk = list(coords.items())[0]
idx_map, row, col = coord
max_activation, file_idx = topk[0]


image, image_trfm = dataset.get_image(file_idx)
display_image_tensor(image, verbose=True)
probed_neuron = (idx_layer, idx_map, row, col)
idx_layer, idx_map, row, col = probed_neuron

Affichage du champ réceptif du neurone surveillé sélectionné, en prenant en référence l'image après transformation.

In [ ]:
batch_input = image_trfm.unsqueeze(dim=0)
output_sizes = get_output_sizes(model_deconv.convnet_features, input_size=batch_input.size())
pixel_space_size = batch_input.size(-2), batch_input.size(-1)
receptive_field = get_receptive_field_in_pixel_space(
    pos=(row, col),
    idx_layer=idx_layer,
    cnn_modules=model_deconv.convnet_features,
    output_sizes=output_sizes,
    pixel_space_size=pixel_space_size
)

((top, left), (bottom, right)) = receptive_field
print("Champ réceptif :", receptive_field, "taille :", right-left+1, "x", bottom-top+1)

Ne pas oublier de redimensioner l'image (avec `image = geo_transforms(image)`) à l'origine de celle de l'entrée pour rendre cohérente la position du champ réception.

In [ ]:
image_resized = geo_transforms(image)
batch_input = batch_input.squeeze(dim=0)
image_resized_receptive_field = T.functional.to_pil_image(image_resized)
img_draw = ImageDraw.Draw(image_resized_receptive_field)
img_draw.rectangle([(left, top), (right, bottom)], outline="red")
display(image_resized_receptive_field)

Déconvolution de l'activation du neurone surveillé sélectionné

In [ ]:
deconvnet.to(device)
start_time = datetime.now()
output_deconv = deconvnet.deconvolution(
    batch_input.to(device),
    idx_layer=idx_layer,
    idx_map=idx_map,
    pos=(row, col),
    clean_feature_map=True,
    return_pos=False,
    verbose=True
    ).detach().cpu()
end_time = datetime.now()
print("Durée de la déconv :", end_time-start_time)

In [ ]:
display_image_tensor(output_deconv)
show_image_tensor(output_deconv, figsize=(2, 2))

In [ ]:
display_image_tensor(output_deconv.clamp(0, 1))
show_image_tensor(output_deconv.clamp(0, 1), figsize=(2, 2))

In [ ]:
output_deconv_cropped = output_deconv[:, top:bottom, left:right]
display_image_tensor(output_deconv_cropped, resize=(200, 200))
display_image_tensor(output_deconv_cropped.clamp(0, 1), resize=(200, 200))
show_image_tensor(output_deconv_cropped, verbose=True)

In [ ]:
output_deconv_in_pixel_space = torch.nn.functional.relu(output_deconv_cropped)
display_image_tensor(output_deconv_in_pixel_space, resize=(200, 200))
display_image_tensor(output_deconv_in_pixel_space.clamp(0, 1), resize=(200, 200))

In [ ]:
output_deconv_in_pixel_space = to_0_1(output_deconv_cropped)
display_image_tensor(output_deconv_in_pixel_space, resize=(200, 200))

In [ ]:
display_rgb_distributions(output_deconv_in_pixel_space.numpy().transpose(1, 2, 0)*255)

In [ ]:
output_deconv_in_pixel_space = inv_normalize(output_deconv_cropped)
display_image_tensor(output_deconv_in_pixel_space, resize=(200, 200))
display_image_tensor(output_deconv_in_pixel_space.clamp(0, 1), resize=(200, 200))

In [ ]:
#display_image_tensor(to_0_255(unnormalize(output_deconv, mean=imagenet_mean, std=imagenet_std)), verbose=True)
output_deconv_in_pixel_space = to_0_1(inv_normalize(output_deconv_cropped))
display_image_tensor(output_deconv_in_pixel_space, resize=(200, 200))

In [ ]:
display_rgb_distributions(output_deconv_in_pixel_space.numpy().transpose(1, 2, 0)*255)

In [ ]:
#output_deconv_in_pixel_space = unnormalize(output_deconv, mean=imagenet_mean, std=imagenet_std)
output_deconv_in_pixel_space = to_0_1(inv_normalize(output_deconv))

batch_input = batch_input.squeeze(dim=0)
output_deconv_receptive_field = T.functional.to_pil_image(output_deconv_in_pixel_space)
img_draw = ImageDraw.Draw(output_deconv_receptive_field)
img_draw.rectangle([(left, top), (right, bottom)], outline="red")
display(output_deconv_receptive_field)

Générer les instructions python utilisant les librairies `torch` et `torchvision` pour créer une image par superposition d'une image contenue dans le tenseur output, dont les coordonnées des coins supérieurs gauche et inférieur droit sont données par les coordonnées des coins supérieur gauche et inférieur droit du champ réceptif fournies par (left, top), (right, bottom), et d'une image de fond contenue dans le tenseur image.

In [ ]:
input_and_deconv = merge_input_and_deconv(image_resized, output_deconv_in_pixel_space, receptive_field, alpha=0.95)
display(input_and_deconv)

In [ ]:
images = [
    image_resized,
    pil_to_tensor(image_resized_receptive_field),
    pil_to_tensor(output_deconv_receptive_field),
    #pil_to_tensor(T.functional.to_pil_image(image_resized).crop((left, top, right, bottom))),
    image_resized[:, top:bottom+1, left:right+1],
    #pil_to_tensor(T.functional.to_pil_image(output_deconv_in_pixel_space).crop((left, top, right, bottom))),
    output_deconv_in_pixel_space[:, top:bottom+1, left:right+1],
    pil_to_tensor(input_and_deconv)
    ]
display_images_list_grid(images, 6)

### Pour l'ensenble des neurones surveillés

In [ ]:
for idx_layer, coords in tqdm(
    topk_activations_by_neuron.items(), total=len(topk_activations_by_neuron), desc="Déconvolutions", unit="couche"
    ):
    for coord, topk in coords.items():
        print(f"=== Layer {idx_layer} neuron {coord} ===")
        idx_map, row, col = coord
        
        for top_value, file_idx in topk:
            print(f"\tTop value : {top_value}, file_idx : {file_idx}")
            image, image_trfm = dataset.get_image(file_idx)
            #display_image_tensor(image, verbose=True)
            probed_neuron = (5, 82, 8, 9)

            # Champ réceptif       
            batch_input = image_trfm.unsqueeze(dim=0)
            output_sizes = get_output_sizes(model_deconv.convnet_features, input_size=batch_input.size())
            pixel_space_size = batch_input.size(-2), batch_input.size(-1)
            receptive_field = get_receptive_field_in_pixel_space(
                pos=(row, col),
                idx_layer=idx_layer,
                cnn_modules=model_deconv.convnet_features,
                output_sizes=output_sizes,
                pixel_space_size=pixel_space_size
            )

            # Drawing the receptive field
            ((top, left), (bottom, right)) = receptive_field
            print("\tChamp réceptif :", receptive_field, "taille :", right-left+1, "x", bottom-top+1)
            image_resized = geo_transforms(image)
            batch_input = batch_input.squeeze(dim=0)
            image_resized_receptive_field = T.functional.to_pil_image(image_resized)
            img_draw = ImageDraw.Draw(image_resized_receptive_field)
            img_draw.rectangle([(left, top), (right, bottom)], outline="red")


            # Déconvolution
            output_deconv = deconvnet.deconvolution(
                batch_input.to(device),
                idx_layer=idx_layer,
                idx_map=idx_map,
                pos=(row, col),
                clean_feature_map=True,
                return_pos=False,
                verbose=False
                ).detach().cpu()
            
            # Transformation to put deconv output in the pixel space
            #output_deconv_in_pixel_space = unnormalize(output_deconv, mean=imagenet_mean, std=imagenet_std)
            output_deconv_in_pixel_space = to_0_1(inv_normalize(output_deconv))
            

            # Drawing the receptive field in the deconv output
            output_deconv_receptive_field = T.functional.to_pil_image(output_deconv_in_pixel_space)
            img_draw = ImageDraw.Draw(output_deconv_receptive_field)
            img_draw.rectangle([(left, top), (right, bottom)], outline="red")

            # Merging
            input_and_deconv = merge_input_and_deconv(
                image_resized, output_deconv_in_pixel_space, receptive_field, alpha=0.95
                )

            # Displaying the images
            images = [
                image_resized,
                pil_to_tensor(image_resized_receptive_field),
                pil_to_tensor(output_deconv_receptive_field),
                #pil_to_tensor(T.functional.to_pil_image(image_resized).crop((left, top, right, bottom))),
                image_resized[:, top:bottom+1, left:right+1],
                #pil_to_tensor(T.functional.to_pil_image(output_deconv_in_pixel_space).crop((left, top, right, bottom))),
                output_deconv_in_pixel_space[:, top:bottom+1, left:right+1],
                pil_to_tensor(input_and_deconv)
                ]
            display_images_list_grid(images, per_rows=6, figsize=(8, 8))